# 02 — GLIMS selection to Parquet for Athena

**Purpose.** Turns the glacier selection from notebook 01 into a Parquet table for AWS Athena. Rolls
the fine-grained GLIMS `geog_area` up into continental buckets (`geog_area_rollup`: Asia,
North America, South America, Europe, Caucausus, Oceania) and drops the polygon geometry, which
doesn't survive the Parquet conversion. The `glims_18k` Athena table built from this file is what the
`sql/` queries join on. The `identify_inference_*` queries use it to pick N glaciers per
`geog_area_rollup`.

## Inputs
- **`src/glims/data/training_samples/glims_18k_bb.shp`** (+ `.shx`, `.dbf`, `.prj`, `.cpg`): the
  output of notebook 01.
- **Environment**: the project's uv environment as the Jupyter kernel (`uv sync`), with geopandas
  and pyarrow. `data_label` in the paths cell isn't used here.

## Outputs
- `glims_18k.parquet`: written to the current working directory (normally `notebooks/pipeline/`).
  Generated output, so don't commit it.

## Manual step afterwards (not in code)
Upload the file to `s3://training-images-metadata-t1-l2-sr/glims_18k/glims_18k.parquet` and make sure
the Glue/Athena table `glims_18k` in the `glacier-view` database (us-west-1) points at that prefix.
This needs AWS credentials with write access to the bucket. See `.agents/references/aws.md` for
bucket regions.

In [1]:
import os

import geopandas as gpd

from glacierview import config

In [2]:
data_label = "localized_time_series_for_training_c02_t1_l2"
glacier_view_dir = str(config.REPO_ROOT)
glims_data_dir = os.path.join(glacier_view_dir,"src","glims","data","training_samples")
glims_path = os.path.join(glims_data_dir,"glims_18k_bb.shp")

In [3]:
df = gpd.read_file(glims_path, index_col = 0)

/Users/mattw/Desktop/projects/GlacierView/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver ESRI Shapefile does not support open option INDEX_COL
  return ogr_read(


In [4]:
region_map = {
    'Himalaya (India, Nepal, Bhutan)': 'Asia',
    'China': 'Asia',
    'Northern Cordillera':'North America',
    'Canada': 'North America',
    'Argentina': 'South America',
    'Asia':'Asia',
    'Alaska':'North America',
    'Caucausus Region': 'Caucausus',
    'Norway': 'Europe',
    'Swiss Alps': 'Europe',
    'Cordillera Blanca, Peru': 'South America',
    'Austrian Alps': 'Europe',
    'Italy': 'Europe',
    'Mongolia': 'Asia',
    'Chile': 'South America',
    'New Zealand': 'Oceania',
    'Siberian Mountains': 'Asia',
    'Central Asia': 'Asia',
    'Prince William Sound': 'North America',
    'Iran': 'Caucausus',
    'Olympic Peninsula': 'North America',
    'Russian Glaciers (exact boundaries of RC to be defined)': 'Asia',
    'Chilean glaciers, between -18 and -56 degrees latitude': 'South America',
    'Turkey': 'Asia',
    'Coropuna, Peru': 'South America',
    'Northern Siberia': 'Asia'
}
df['geog_area_rollup'] = df.geog_area.map(region_map)

In [5]:
df['geog_area_rollup'].value_counts()

geog_area_rollup
Asia             12625
North America     3704
South America      872
Europe             602
Caucausus          255
Oceania             35
Name: count, dtype: int64

In [6]:
df = df.drop(['geometry'], axis = 1) #geometry polygon format doesn't behave well when converted to parquet

In [7]:
df.to_parquet("glims_18k.parquet")